In [2]:
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

API rate limits take into account client identity to determine the level of access.
Stronger forms of identification result in a higher limit, such as running in Wikimedia Cloud Services (WMCS) or authenticating requests.
The highest limits require running in WMCS, community bot approval, or being well-known to the Wikimedia Foundation.
Further limits or additional best practices may apply to specific APIs.

In [16]:
# Wikipedia bot with a compliant user-agent

import requests

# Wikipedia bot/user-agent policy requires identifying the bot clearly and providing contact info.
# Use the official MediaWiki API instead of scraping page HTML when possible.
headers = {
    "User-Agent": "CSS_ProjectBot/1.0 (https://github.com/Stampe04/CSS_Project; fwaggot.player@gmail.com)",
    "From": "fwaggot.player@gmail.com",
    "Accept": "application/json"
}

url = "https://en.wikipedia.org/w/api.php"
params = {
    "action": "query",
    "list": "allusers",
    "augroup": "sysop",
    "aulimit": "max",
    "format": "json"
}

admins = []

while True:
    response = requests.get(url, headers=headers, params=params, timeout=10)
    response.raise_for_status()
    data = response.json()

    admins.extend([user["name"] for user in data["query"]["allusers"]])

    if "continue" not in data:
        break

    params.update(data["continue"])

print(f"Administrators: {len(admins)}")
for admin in admins[::10]:
    print("-", admin)


# save the list of admins to a CSV file for later analysis
admins_df = pd.DataFrame(admins, columns=["Admin"])
admins_df.to_csv("wikipedia_admins.csv", index=False)
print("Saved list of administrators to wikipedia_admins.csv")

Administrators: 811
- 28bytes
- Acalamari
- Ajpolino
- Amorymeltzer
- Antandrus
- Atlant
- BDD
- BethNaught
- Black Kite
- Brian Kendig
- CJCurrie
- Canadian Paul
- Cburnett
- Chris G
- Colin M
- Cryptic
- DYKUpdateBot
- DatGuy
- Deb
- DerHexer
- Doczilla
- Drmies
- Edcolins
- EncMstr
- Extraordinary Writ
- Feezo
- Frank
- Galobtter
- Gentgeen
- GoldRingChip
- Ground Zero
- Harrias
- Hiding
- Ianblair23
- Isabelle Belato
- JIP
- Jamesofur
- Jesse Viviano
- John K
- K6ka
- Keilana
- KrakatoaKatie
- LFaraone
- Lexicon
- Lustiger seth
- Mailer diablo
- Mattythewhite
- Metamagician3000
- Mike Rosoft
- Mjroots
- Morwen
- NativeForeigner
- NinjaRobotPirate
- Ohnoitsjamie
- PFHLai
- PedanticallySpeaking
- Pinkville
- Prolog
- Ragesoss
- RegentsPark
- Rjjiii
- Rogerd
- SQL
- SarekOfVulcan
- Sdrqaz
- Sesel
- SimonP
- Smartse
- Spellcast
- Steven Walling
- TadejM
- The Anome
- The ed17
- TigerShark
- Tom Morris
- U193581
- Valfontis
- Visviva
- Wehwalt
- Wugapodes
- Yaris678
- Zzyzx11
Saved list

In [15]:
# Example: fetch structured metrics for multiple admins using the MediaWiki API
# This is more reliable than scraping the HTML page, because the API returns fields directly.

if admins:
    target_admins = admins[:5]  # Fetch metrics for the first 5 admins instead of just one
else:
    target_admins = ["Example"]

params = {
    "action": "query",
    "list": "users",
    "ususers": "|".join(target_admins),
    "usprop": "editcount|groups|registration|rights|blockinfo",
    "format": "json"
}

response = requests.get(url, headers=headers, params=params, timeout=10)
response.raise_for_status()

user_data = response.json()
users = user_data["query"]["users"]

for user in users:
    print(f"\nAdmin metrics for: {user.get('name')}")
    print("editcount:", user.get("editcount"))
    print("registration:", user.get("registration"))
    print("groups:", user.get("groups"))
    print("rights:", user.get("rights"))
    print("blocked:", user.get("blocked"))


Admin metrics for: 28bytes
editcount: 32812
registration: 2006-12-14T23:17:44Z
groups: ['autoreviewer', 'bureaucrat', 'sysop', '*', 'user', 'autoconfirmed']
rights: ['autopatrol', 'move-subpages', 'suppressredirect', 'tboverride', 'noratelimit', 'checkuser-temporary-account', 'checkuser-temporary-account-auto-reveal', 'oathauth-verify-user', 'oathauth-view-log', 'override-antispoof', 'templateeditor', 'changetags', 'extendedconfirmed', 'deleterevision', 'deletelogentry', 'editcontentmodel', 'block', 'createaccount', 'delete', 'deletedhistory', 'deletedtext', 'undelete', 'editinterface', 'editsitejson', 'edituserjson', 'import', 'move', 'move-rootuserpages', 'move-categorypages', 'patrol', 'protect', 'editprotected', 'rollback', 'upload', 'reupload', 'reupload-shared', 'unwatchedpages', 'autoconfirmed', 'editsemiprotected', 'ipblock-exempt', 'blockemail', 'markbotedits', 'apihighlimits', 'browsearchive', 'movefile', 'mergehistory', 'managechangetags', 'deletechangetags', 'abusefilter-r